In [1]:
import os
import requests
import zipfile
import tarfile

data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)
zip_path = os.path.join(data_dir, "lesson6.tar.zip")

url = "https://www.dropbox.com/scl/fi/rihfngx4ju5pzjzjj7u9z/lesson6.tar.zip?rlkey=rct9a9bo8euqgshrk8wiq2orh&dl=1"


print("Downloading dataset...")
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(zip_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
print("Download completed!")


print("Extracting zip...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(data_dir)
print("Zip extracted!")


tar_files = [f for f in os.listdir(data_dir) if f.endswith(".tar")]
for tar_file in tar_files:
    tar_path = os.path.join(data_dir, tar_file)
    print(f"Extracting {tar_file} ...")
    with tarfile.open(tar_path, 'r') as tar_ref:
        tar_ref.extractall(data_dir)
print("All extraction completed!")

Download completed!
Extracting zip...
Zip extracted!
Extracting lesson6.tar ...
All extraction completed!


/tmp/ipython-input-1067084780.py:34: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar_ref.extractall(data_dir)


In [2]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.models import Transformer, Pooling, Dense
from sentence_transformers.losses import CosineSimilarityLoss
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
import torch
from datasets import Dataset
from tqdm.auto import tqdm
from api_utils import Utils

> **To fine-tuning a model using `sentence_transformers`, you can read the docs here:** https://sbert.net/docs/sentence_transformer/training_overview.html#>

# Dataset

In [3]:
with open("./data/training.txt", 'r') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        if i <= 3:
            print(line)

Apr 15 2013 09:36:50: %ASA-4-106023: Deny tcp src dmz:10.1.2.30/63016 dst outside:192.0.0.8/53 by access-group "acl_dmz" [0xe3aab522, 0x0] ^ Apr 15 2013 09:36:50: %ASA-4-106023: Deny tcp src dmz:10.1.2.30/63016 dst outside:192.0.0.8/53 by access-group "acl_dmz" [0xe3aab522, 0x0] ^ 1.0

Apr 15 2013 09:36:50: %ASA-4-106023: Deny tcp src dmz:10.1.2.30/63016 dst outside:192.0.0.8/53 type 3, code 0, by access-group "acl_dmz" [0xe3aab522, 0x0] ^ Apr 15 2013 09:36:50: %ASA-4-106023: Deny tcp src dmz:10.1.2.30/63016 dst outside:192.0.0.8/53 by access-group "acl_dmz" [0xe3aab522, 0x0] ^ 0.9

Apr 15 2014 09:34:34 EDT: %ASA-session-5-106100: access-list acl_in permitted tcp inside/10.1.2.16(2241) -> outside/192.0.0.89(2000) hit-cnt 1 first hit [0x71a87d94, 0x0] ^ Apr 15 2013 09:36:50: %ASA-4-106023: Deny tcp src dmz:10.1.2.30/63016 dst outside:192.0.0.8/53 by access-group "acl_dmz" [0xe3aab522, 0x0] ^ 0.8

Apr 24 2013 16:00:28 INT-FW01 : %ASA-6-106100: access-list inside denied udp inside/172.29.

In [4]:
def load_dataset(file_path):
    text1, text2, label = [], [], []
    with open(file_path, 'r') as f:
        lines = f.readlines()
        for line in lines:
            line = line.strip()
            if line:
                t1, t2, similarity = line.split('^')
                text1.append(t1.strip())
                text2.append(t2.strip())
                label.append(float(similarity.strip()))

    return Dataset.from_dict(
        {"text1": text1,
         "text2": text2,
         "label": label}
    )

In [5]:
train_dataset = load_dataset('./data/training.txt')
train_dataset

Dataset({
    features: ['text1', 'text2', 'label'],
    num_rows: 26
})

# Build Model

In [6]:
def build_model(base_model='distilbert-base-uncased'):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    embedding_model = Transformer(base_model)
    pooling_model = Pooling(
        embedding_model.get_word_embedding_dimension(),
        pooling_mode_mean_tokens=True,
        pooling_mode_cls_token=False,
        pooling_mode_max_tokens=False
    )
    dense_model = Dense(
        in_features=pooling_model.get_sentence_embedding_dimension(),
        out_features=256,
        activation_function=torch.nn.Tanh()
    )

    model = SentenceTransformer(modules=[embedding_model, pooling_model, dense_model], device=device)
    return model

In [7]:
model = build_model()
model

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'DistilBertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 256, 'bias': True, 'activation_function': 'torch.nn.modules.activation.Tanh'})
)

In [8]:
for name, param in model[0].named_parameters():
  print(f"{name:<60}: {param.shape}")

auto_model.embeddings.word_embeddings.weight                : torch.Size([30522, 768])
auto_model.embeddings.position_embeddings.weight            : torch.Size([512, 768])
auto_model.embeddings.LayerNorm.weight                      : torch.Size([768])
auto_model.embeddings.LayerNorm.bias                        : torch.Size([768])
auto_model.transformer.layer.0.attention.q_lin.weight       : torch.Size([768, 768])
auto_model.transformer.layer.0.attention.q_lin.bias         : torch.Size([768])
auto_model.transformer.layer.0.attention.k_lin.weight       : torch.Size([768, 768])
auto_model.transformer.layer.0.attention.k_lin.bias         : torch.Size([768])
auto_model.transformer.layer.0.attention.v_lin.weight       : torch.Size([768, 768])
auto_model.transformer.layer.0.attention.v_lin.bias         : torch.Size([768])
auto_model.transformer.layer.0.attention.out_lin.weight     : torch.Size([768, 768])
auto_model.transformer.layer.0.attention.out_lin.bias       : torch.Size([768])
auto_mod

In [9]:
# Freeze all layers
for param in model[0].auto_model.parameters():
    param.requires_grad = False

# Unfreeze few last layers
for i in [4, 5]:
  for param in model[0].auto_model.transformer.layer[i].parameters():
      param.requires_grad = True

In [10]:
for name, param in model[0].auto_model.named_parameters():
    if param.requires_grad:
        print("Trainable:", name)

Trainable: transformer.layer.4.attention.q_lin.weight
Trainable: transformer.layer.4.attention.q_lin.bias
Trainable: transformer.layer.4.attention.k_lin.weight
Trainable: transformer.layer.4.attention.k_lin.bias
Trainable: transformer.layer.4.attention.v_lin.weight
Trainable: transformer.layer.4.attention.v_lin.bias
Trainable: transformer.layer.4.attention.out_lin.weight
Trainable: transformer.layer.4.attention.out_lin.bias
Trainable: transformer.layer.4.sa_layer_norm.weight
Trainable: transformer.layer.4.sa_layer_norm.bias
Trainable: transformer.layer.4.ffn.lin1.weight
Trainable: transformer.layer.4.ffn.lin1.bias
Trainable: transformer.layer.4.ffn.lin2.weight
Trainable: transformer.layer.4.ffn.lin2.bias
Trainable: transformer.layer.4.output_layer_norm.weight
Trainable: transformer.layer.4.output_layer_norm.bias
Trainable: transformer.layer.5.attention.q_lin.weight
Trainable: transformer.layer.5.attention.q_lin.bias
Trainable: transformer.layer.5.attention.k_lin.weight
Trainable: trans

# Training

In [11]:
loss = CosineSimilarityLoss(model)
loss

CosineSimilarityLoss(
  (model): SentenceTransformer(
    (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'DistilBertModel'})
    (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
    (2): Dense({'in_features': 768, 'out_features': 256, 'bias': True, 'activation_function': 'torch.nn.modules.activation.Tanh'})
  )
  (loss_fct): MSELoss()
  (cos_score_transformation): Identity()
)

In [12]:
evaluator = EmbeddingSimilarityEvaluator(
    sentences1 = train_dataset["text1"],
    sentences2 = train_dataset["text2"],
    scores = train_dataset["label"],
    main_similarity = SimilarityFunction.COSINE
)

# evaluate model before fine-tuning
evaluator(model)

{'pearson_cosine': 0.38841928651818447, 'spearman_cosine': 0.790074820616131}

In [13]:
# check whether can use bfloat16 to train instead of float32 or float16
# if bfloat16 not supported, then use float16
use_bf16 = torch.cuda.is_bf16_supported()
use_bf16

True

In [14]:
# to clean folder models/ before training
import shutil
shutil.rmtree('models', ignore_errors=True)

In [15]:
training_args = SentenceTransformerTrainingArguments(
    output_dir='models/',  # store checkpoints in this folder
    save_strategy='epoch', # store a checkpoint after each epoch
    save_total_limit=3,    # folder retains only last 3 checkpoints
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-4,
    bf16=use_bf16,
    fp16=not use_bf16,
    eval_strategy='epoch',
    load_best_model_at_end=True,  # store best checkpoint in `trainer.model`
    metric_for_best_model='spearman_cosine',
    greater_is_better=True,  # the greater the metric the better
    logging_dir=None,
    logging_strategy='epoch',
    report_to=[],
    dataloader_pin_memory = torch.cuda.is_available()  # to avoid warning
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=train_dataset,
    loss=loss,
    evaluator=evaluator
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [16]:
trainer.train(resume_from_checkpoint=None)

Epoch,Training Loss,Validation Loss,Pearson Cosine,Spearman Cosine
1,3.320600,3.207210,0.334136,0.800458
2,3.220000,3.162734,0.316075,0.809925
3,3.182500,3.169095,0.328313,0.832715
4,3.182200,3.145668,0.347619,0.859712
5,3.151900,3.131986,0.368853,0.868626
6,3.142700,3.132746,0.384890,0.880048
7,3.150700,3.131066,0.394029,0.899683
8,3.140700,3.127381,0.400002,0.920877
9,3.144700,3.124569,0.402360,0.922122
10,3.143400,3.123381,0.402756,0.922122


TrainOutput(global_step=10, training_loss=3.177937865257263, metrics={'train_runtime': 63.4464, 'train_samples_per_second': 4.098, 'train_steps_per_second': 0.158, 'total_flos': 0.0, 'train_loss': 3.177937865257263, 'epoch': 10.0})

In [17]:
best_ckpt = trainer.state.best_model_checkpoint
best_metric_name = trainer.args.metric_for_best_model
best_metric_value = trainer.state.best_metric

print(f"Best model found at: {best_ckpt}")
print(f"Metric ({best_metric_name}): {best_metric_value:.6f}")

Best model found at: models/checkpoint-9
Metric (spearman_cosine): 0.922122


In [18]:
trainer.model.save('./models/best_model')

# Using Pretrained Model to create Embedding

In [19]:
best_model = SentenceTransformer('./models/best_model')
best_model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'DistilBertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 256, 'bias': True, 'activation_function': 'torch.nn.modules.activation.Tanh'})
)

In [20]:
EMBEDDING_DIM = best_model.get_sentence_embedding_dimension()
EMBEDDING_DIM

256

In [21]:
def load_logs(log_path):
    logs = []
    with open(log_path, 'r') as f:
        lines = f.readlines()
        for line in lines:
            line = line.strip()
            if line:
                logs.append(line)

    print(f"Loaded successfully {len(logs)} logs")
    return logs

In [22]:
logs = load_logs('./data/sample.log')

Loaded successfully 90 logs


In [23]:
def upsert_logs(logs, index, embedding_model):
    ids = [str(i) for i in range(len(logs))]
    embs = embedding_model.encode(logs)
    meta_logs = [{"log": log} for log in logs]

    all_zip = zip(ids, embs, meta_logs)
    records = list(
        map(lambda x: {'id':x[0], 'values':x[1].tolist(), 'metadata':x[2]}, all_zip)
    )
    index.upsert(records)

In [24]:
log_index = Utils.create_index('log-index', dimension=EMBEDDING_DIM)

Index 'log-index' already exists → deleting ...
Deleted 'log-index' successfully!
Creating index 'log-index' ...
Index 'log-index' created successfully!


In [25]:
upsert_logs(logs, log_index, best_model)

# Anomaly Detection

In [26]:
from rich import print

In [27]:
def query_logs(good_log, index, embedding_model, threshold=0.7):
  results = index.query(
      vector=embedding_model.encode(good_log).tolist(),
      top_k=100,
      include_metadata=True
  )

  matches = results.matches

  for m in matches:
      score = m.score
      log = m.metadata.get("log", "")

      color = "bold green" if score >= threshold else "bold red"
      print(f"[{color}]{score:.4f} — {log}[/{color}]")
      print('---')

  print()
  num_anomalies = sum(m.score < threshold for m in matches)
  print(f"Found {num_anomalies} anomalies out of {len(matches)} results.")

In [28]:
good_log = logs[0]
good_log

'Apr 15 2013 09:36:50: %ASA-4-106023: Deny tcp src dmz:10.1.2.30/63016 dst outside:192.0.0.8/53 by access-group "acl_dmz" [0xe3aab522, 0x0]'

In [30]:
query_logs(good_log, log_index, best_model, threshold=0.75)

1.0002 — Apr 15 2013 09:36:50: %ASA-4-106023: Deny tcp src dmz:10.1.2.30/63016 dst outside:192.0.0.8/53 by 
access-group "acl_dmz" [0xe3aab522, 0x0]

---

0.9845 — Apr 15 2013 09:36:50: %ASA-4-106023: Deny tcp src dmz:10.1.2.30/63016 dst outside:192.0.0.8/53 type 3, 
code 0, by access-group "acl_dmz" [0xe3aab522, 0x0]

---

0.9603 — Apr 30 2013 09:23:40: %ASA-4-106023: Deny tcp src outside:192.0.2.126/53638 dst inside:10.0.0.132/8111 by 
access-group "acl_out" [0x71761f18, 0x0]

...

0.4221 — dec 31, 2021 09:18:59: %ASA-4-434005: seg fault detected in the matrix

---

0.3727 — Jan 14 2015 13:16:14: %ASA-4-338002: Dynamic Filter permitted black listed TCP traffic from 
inside:10.1.1.45/6798 (192.88.99.1/7890) to outside:192.88.99.129/80 (192.88.99.129/80), destination 192.88.99.129 
resolved from dynamic list: bad.example.com

---

Found 30 anomalies out of 90 results.